# Testing
### Aufenthaltsdauer beim Wegzug nach auswärts nach Alter, Geschlecht, Herkunft und Stadtkreis

Aufenthaltsdauer beim Wegzug nach auswärts des Bevölkerungsbestandes nach Alter, Geschlecht, Herkunft und Stadtkreis

Datum: 15.03.2023




### Importiere die notwendigen Packages

In [37]:
#%pip install geopandas altair fiona requests folium mplleaflet contextily seaborn datetime plotly leafmap

In [38]:
import altair as alt
import datetime
import folium 
import geopandas as gpd
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#import pivottablejs
#from pivottablejs import pivot_ui
import plotly.express as px
import requests
import seaborn as sns


In [39]:
SSL_VERIFY = False
# evtl. SSL_VERIFY auf False setzen wenn die Verbindung zu https://www.gemeinderat-zuerich.ch nicht klappt (z.B. wegen Proxy)
# Um die SSL Verifikation auszustellen, bitte die nächste Zeile einkommentieren ("#" entfernen)
# SSL_VERIFY = False

In [40]:
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings()

Definiere Settings. Hier das Zahlenformat von Float-Werten (z.B. *'{:,.2f}'.format* mit Komma als Tausenderzeichen), 

In [41]:
#pd.options.display.float_format = lambda x : '{:,.1f}'.format(x) if (np.isnan(x) | np.isinf(x)) else '{:,.0f}'.format(x) if int(x) == x else '{:,.1f}'.format(x)
pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 15)

### Zeitvariabeln
Bestimme den aktuellst geladenen Monat. Hier ist es der Stand vor 2 Monaten. 
Bestimme noch weitere evt. sinnvolle Zeitvariabeln.

Zum Unterschied zwischen import datetime und from datedtime import datetime, siehe https://stackoverflow.com/questions/15707532/import-datetime-v-s-from-datetime-import-datetime

Zuerst die Zeitvariabeln als Strings

In [42]:
#today_date = datetime.date.today()
#date_time = datetime.datetime.strptime(date_time_string, '%Y-%m-%d %H:%M')
now = datetime.date.today()
date_today = now.strftime("%Y-%m-%d")
year_today = now.strftime("%Y")
month_today = now.strftime("%m")
day_today = now.strftime("%d")


Und hier noch die Zeitvariabeln als Integers:
- `aktuellesJahr`
- `aktuellerMonat`: Der gerade jetzt aktuelle Monat
- `selectedMonat`: Der aktuellste Monat in den Daten. In der Regel zwei Monate her.

In [43]:
#now = datetime.now() 
int_times = now.timetuple()

aktuellesJahr = int_times[0]
aktuellerMonat = int_times[1]
selectedMonat = int_times[1]-2

print(aktuellesJahr, 
      aktuellerMonat,
    'datenstand: ', selectedMonat,
     int_times)


2024 3 datenstand:  1 time.struct_time(tm_year=2024, tm_mon=3, tm_mday=10, tm_hour=0, tm_min=0, tm_sec=0, tm_wday=6, tm_yday=70, tm_isdst=-1)


Berechne die Variable Epoche um später das SAS-Datum in ein Unix-Datum umzuwandeln. Bei SAS beginnt die Epoche am 1.1.1960. Bei Unix am 1.1.1970.
Diese Variable wird beim CSV-Import benötigt.

In [44]:
epoch = datetime.datetime(1960, 1, 1)

### Setze einige Pfadvariabeln

- Der Packagename ist eigentlich der **Verzeichnisname** unter dem die Daten und Metadaten auf der Dropzone abgelegt werden.
- Definiert wird er bei SASA-Prozessen auf dem **Produkte-Sharepoint ([Link](https://kollaboration.intranet.stzh.ch/orga/ssz-produkte/Lists/SASA_Outputs/PersonalViews.aspx?PageView=Personal&ShowWebPart={6087A3E7-8AC8-40BA-8278-DECFACE124FF}))**.
- Der Packagename wird auf CKAN teil der URL, daher ist die exakte Schreibweise wichtig.

Beachte: im Packagename müssen alle Buchstaben **klein** geschrieben werden. Dies weil CKAN aus grossen kleine Buchstaben macht.

**BITTE HIER ANPASSEN**

In [45]:
package_name = "bev_aufenthaltsdauer_wegzug_alter_geschlecht_herkunft_stadtkreis_od5251"

In [46]:
dataset_name = "BEV525OD5251.csv"

**Statische Pfade in DWH-Dropzones**

In [47]:
dropzone_path_integ = r"\\szh\ssz\applikationen\OGD_Dropzone\INT_DWH"

In [48]:
dropzone_path_prod = r"\\szh\ssz\applikationen\OGD_Dropzone\DWH"

**Statische Pfade CKAN-URLs**

In [49]:
ckan_integ_url ="https://data.integ.stadt-zuerich.ch/dataset/int_dwh_"

In [50]:
ckan_prod_url ="https://data.stadt-zuerich.ch/dataset/"

### Checke die Metadaten auf der CKAN INTEG- oder PROD-Webseite

Offenbar lassen sich aktuell im Markdownteil keine Variabeln ausführen, daher gehen wir wie unten gezeigt vor. Siehe dazu: https://data-dive.com/jupyterlab-markdown-cells-include-variables
Instead of setting the cell to Markdown, create Markdown from withnin a code cell! We can just use python variable replacement syntax to make the text dynamic

In [51]:
from IPython.display import Markdown as md

In [52]:
md(" **1. Dataset auf INTEG-Datakatalog:** Link {} ".format(ckan_integ_url+package_name))

 **1. Dataset auf INTEG-Datakatalog:** Link https://data.integ.stadt-zuerich.ch/dataset/int_dwh_bev_aufenthaltsdauer_wegzug_alter_geschlecht_herkunft_stadtkreis_od5251 

In [53]:
md(" **2. Dataset auf PROD-Datakatalog:** Link {} ".format(ckan_prod_url+package_name))

 **2. Dataset auf PROD-Datakatalog:** Link https://data.stadt-zuerich.ch/dataset/bev_aufenthaltsdauer_wegzug_alter_geschlecht_herkunft_stadtkreis_od5251 

### Importiere einen Datensatz 

Definiere zuerst folgende Werte:
1) Kommt der Datensatz von PROD oder INTEG?
2) Beziehst Du den Datensatz direkt ab der DROPZONE oder aus dem INTERNET?

In [54]:
#Die Datasets sind nur zum Testen auf INT-DWH-Dropzone. Wenn der Test vorbei ist, sind sie auf PROD. 
# Über den Status kann man einfach switchen

status = "integ"; #prod vs something else
data_source = "web"; #dropzone vs something else
print(status+" - "+ data_source)

integ - web


In [55]:
# Filepath
if status == "prod":
    if data_source == "dropzone":
            fp = dropzone_path_prod+"\\"+ package_name +"\\"+dataset_name
            print("fp lautet:"+fp)
    else:
        #fp = r"https://data.stadt-zuerich.ch/dataset/bau_neubau_whg_bausm_rinh_geb_projstatus_quartier_seit2009_od5011/download/BAU501OD5011.csv"
        fp = ckan_prod_url+package_name+'/download/'+dataset_name
        print("fp lautet:"+fp)
else:
    if data_source == "dropzone":
        fp = dropzone_path_integ+"\\"+ package_name +"\\"+dataset_name
        print("fp lautet:"+fp)
    else:
        #fp = r"https://data.stadt-zuerich.ch/dataset/bau_neubau_whg_bausm_rinh_geb_projstatus_quartier_seit2009_od5011/download/BAU501OD5011.csv"
        fp = ckan_integ_url+package_name+'/download/'+dataset_name
        print("fp lautet:"+fp)


fp lautet:https://data.integ.stadt-zuerich.ch/dataset/int_dwh_bev_aufenthaltsdauer_wegzug_alter_geschlecht_herkunft_stadtkreis_od5251/download/BEV525OD5251.csv


Beachte, wie das SAS Datum (ohne Format) in ein UNIX Datum umgerechnet und als Datumsformat dargestellt wird! Siehe dazu `https://stackoverflow.com/questions/26923564/convert-sas-numeric-to-python-datetime`

In [56]:
# Read the data
if data_source == "dropzone":
    data2betested = pd.read_csv(
        fp
        , sep=','
        ,parse_dates=['StichtagDatJahr']
        ,low_memory=False
    )
    print("dropzone")
else:
    r = requests.get(fp, verify=False)  
    r.encoding = 'utf-8'
    data2betested = pd.read_csv(
        io.StringIO(r.text)
        ,parse_dates=['StichtagDatJahr']
        # KONVERTIERE DAS SAS DATUM IN EIN UNIXDATUM UND FORMATIERE ES
        #, date_parser=lambda s: epoch + datetime.timedelta(days=int(s))
        ,low_memory=False)
    print("web")

data2betested.dtypes

web


StichtagDatJahr             datetime64[ns]
AlterV20ueber80Sort_noDM             int64
AlterV20ueber80Cd_noDM               int64
AlterV20ueber80Kurz_noDM            object
SexCd                                int64
SexKurz                             object
HerkunftCd                           int64
HerkunftLang                        object
KreisCd                              int64
KreisLang                           object
AufDauerP25                        float64
AufDauerMedian                     float64
AufDauerP75                        float64
AufDauerMittel                     float64
dtype: object

Berechne weitere Attribute falls notwendig

In [57]:
data2betested = (
    data2betested
    .copy()
    .assign(
        #Aktualisierungs_Datum_str= lambda x: x.Aktualisierungs_Datum.astype(str),
        Jahr = lambda x: x.StichtagDatJahr,
        Jahr_str = lambda x: x.StichtagDatJahr.astype(str),
        Jahr_nbr = lambda x: x.Jahr.dt.year,
    )
    .sort_values('StichtagDatJahr', ascending=False)
    )
data2betested.dtypes

StichtagDatJahr             datetime64[ns]
AlterV20ueber80Sort_noDM             int64
AlterV20ueber80Cd_noDM               int64
AlterV20ueber80Kurz_noDM            object
SexCd                                int64
SexKurz                             object
HerkunftCd                           int64
HerkunftLang                        object
KreisCd                              int64
KreisLang                           object
AufDauerP25                        float64
AufDauerMedian                     float64
AufDauerP75                        float64
AufDauerMittel                     float64
Jahr                        datetime64[ns]
Jahr_str                            object
Jahr_nbr                             int64
dtype: object

Minimales und maximales Jahr im Datensatz

In [58]:
data_max_date = str(max(data2betested.Jahr).year)
data_min_date = str(min(data2betested.Jahr).year)

print(f"Die Daten haben ein Minimumjahr von {data_min_date} und ein Maximumjahr von {data_max_date}")

Die Daten haben ein Minimumjahr von 1993 und ein Maximumjahr von 2023


### Einfache Datentests

 - 1) Zeige eine kurze Vorschau der importierten Daten
 - 2) Weise die Datentypen aus
 - 3) Zeige die Shape (Umfang) des Datensatzes an

In [59]:
#data2betested.head(6)

In [60]:
data2betested.dtypes

StichtagDatJahr             datetime64[ns]
AlterV20ueber80Sort_noDM             int64
AlterV20ueber80Cd_noDM               int64
AlterV20ueber80Kurz_noDM            object
SexCd                                int64
SexKurz                             object
HerkunftCd                           int64
HerkunftLang                        object
KreisCd                              int64
KreisLang                           object
AufDauerP25                        float64
AufDauerMedian                     float64
AufDauerP75                        float64
AufDauerMittel                     float64
Jahr                        datetime64[ns]
Jahr_str                            object
Jahr_nbr                             int64
dtype: object

In [61]:
data2betested.shape

(7270, 17)

Beschreibe einzelne Attribute

In [62]:
data2betested.describe()

,AlterV20ueber80Sort_noDM,AlterV20ueber80Cd_noDM,SexCd,HerkunftCd,KreisCd,AufDauerP25,AufDauerMedian,AufDauerP75,AufDauerMittel,Jahr_nbr
count,7270,7270,7270,7270,7270,7270,7270,7270,7270,7270
mean,3,3,2,1,7,6,12,20,14,2008
std,1,1,1,0,3,11,16,20,15,9
min,1,1,1,1,1,0,0,0,0,1993
25%,2,2,1,1,4,0,1,5,4,2000
50%,3,3,2,1,7,1,3,10,7,2008
75%,4,4,2,2,10,4,15,32,19,2016
max,5,5,2,2,12,75,86,94,80,2023


Wie viele Nullwerte gibt es im Datensatz?

In [63]:
data2betested.isnull().sum()

StichtagDatJahr             0
AlterV20ueber80Sort_noDM    0
AlterV20ueber80Cd_noDM      0
AlterV20ueber80Kurz_noDM    0
SexCd                       0
SexKurz                     0
HerkunftCd                  0
HerkunftLang                0
KreisCd                     0
KreisLang                   0
AufDauerP25                 0
AufDauerMedian              0
AufDauerP75                 0
AufDauerMittel              0
Jahr                        0
Jahr_str                    0
Jahr_nbr                    0
dtype: int64

Welches sind die Quartiere ohne Werte bei AnzBestWir?

In [64]:
data2betested[np.isnan(data2betested.AufDauerMittel)]

,StichtagDatJahr,AlterV20ueber80Sort_noDM,AlterV20ueber80Cd_noDM,AlterV20ueber80Kurz_noDM,SexCd,SexKurz,HerkunftCd,...,AufDauerP25,AufDauerMedian,AufDauerP75,AufDauerMittel,Jahr,Jahr_str,Jahr_nbr


### Verwende das Datum als Index

While we did already parse the `datetime` column into the respective datetime type, it currently is just a regular column. 
**To enable quick and convenient queries and aggregations, we need to turn it into the index of the DataFrame**

In [65]:
data2betested = data2betested.set_index("StichtagDatJahr")

In [66]:
data2betested.info()
data2betested.index.year.unique()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7270 entries, 2023-01-01 to 1993-01-01
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   AlterV20ueber80Sort_noDM  7270 non-null   int64         
 1   AlterV20ueber80Cd_noDM    7270 non-null   int64         
 2   AlterV20ueber80Kurz_noDM  7270 non-null   object        
 3   SexCd                     7270 non-null   int64         
 4   SexKurz                   7270 non-null   object        
 5   HerkunftCd                7270 non-null   int64         
 6   HerkunftLang              7270 non-null   object        
 7   KreisCd                   7270 non-null   int64         
 8   KreisLang                 7270 non-null   object        
 9   AufDauerP25               7270 non-null   float64       
 10  AufDauerMedian            7270 non-null   float64       
 11  AufDauerP75               7270 non-null   float64       
 12  Au

Int64Index([2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010,
            2009, 2008, 2007, 2006, 2005, 2004, 2003, 2002, 2001, 2000, 1999, 1998, 1997, 1996,
            1995, 1994, 1993],
           dtype='int64', name='StichtagDatJahr')

### Einfache Visualisierungen zur Plausi

Exploriere die Daten mit Pivottable.JS

In [67]:
#pivot_ui(data2betested)

### Zeitpunkte und Zeiträume abfragen

A particular powerful feature of the Pandas DataFrame is its indexing capability that also works using time-based entities, such as dates and times. We have already created the index above, so let's put it to use.

In [71]:
data2betested.loc[data_max_date].head(2)
#data2betested.loc["2021-10-31":"2021-11-30"].head(2)

,AlterV20ueber80Sort_noDM,AlterV20ueber80Cd_noDM,AlterV20ueber80Kurz_noDM,SexCd,SexKurz,HerkunftCd,HerkunftLang,...,AufDauerP25,AufDauerMedian,AufDauerP75,AufDauerMittel,Jahr,Jahr_str,Jahr_nbr
StichtagDatJahr,,,,,,,,,,,,,,,
2023-01-01,5,5,80 u. älter,2,W,2,Ausländer*in,...,22,43,55,39,2023-01-01,2023-01-01,2023
2023-01-01,2,2,20-39,2,W,1,Schweizer*in,...,2,5,11,8,2023-01-01,2023-01-01,2023


### Visualisierungen nach Zeitausschnitten

In [72]:
data2betested.reset_index().columns

Index(['StichtagDatJahr', 'AlterV20ueber80Sort_noDM', 'AlterV20ueber80Cd_noDM',
       'AlterV20ueber80Kurz_noDM', 'SexCd', 'SexKurz', 'HerkunftCd', 'HerkunftLang', 'KreisCd',
       'KreisLang', 'AufDauerP25', 'AufDauerMedian', 'AufDauerP75', 'AufDauerMittel', 'Jahr',
       'Jahr_str', 'Jahr_nbr'],
      dtype='object')

#### Mittlere Aufenthaltsdauer nach Altersgruppe

In [73]:
myAgg = data2betested.reset_index()\
    .groupby(['StichtagDatJahr','AlterV20ueber80Sort_noDM', 'AlterV20ueber80Kurz_noDM']) \
    .agg(mean_WBev=('AufDauerMittel', 'mean')) \
    .sort_values('AlterV20ueber80Sort_noDM', ascending=True) 

myAgg.reset_index().head(3)

,StichtagDatJahr,AlterV20ueber80Sort_noDM,AlterV20ueber80Kurz_noDM,mean_WBev
0,1993-01-01,1,0-19,3
1,1997-01-01,1,0-19,4
2,2002-01-01,1,0-19,4


In [75]:
myTitle="Mittlere Aufenthaltsdauer nach Altersgruppe, "+data_min_date + "- "+data_max_date

highlight = alt.selection(type='single', on='mouseover',
                          fields=['AlterV20ueber80Kurz'], nearest=True)
#x='date:StichtagDatJahr',
base = alt.Chart(myAgg.reset_index().query('mean_WBev>0').sort_values('AlterV20ueber80Sort_noDM', ascending=True), title=myTitle).encode(
    x=alt.X('StichtagDatJahr', axis=alt.Axis(title='Stadtkreis'))# , axis=alt.Axis(format='%', title='percentage')
    , y=alt.X('mean_WBev', axis=alt.Axis(title='Mean Aufenthaltsdauer'))
    , color=alt.Color('AlterV20ueber80Kurz_noDM', legend=alt.Legend(title="Altersgruppen", orient="right"))  
    ,tooltip=['StichtagDatJahr', 'AlterV20ueber80Kurz_noDM','mean_WBev']    
)
points = base.mark_circle().encode(
    opacity=alt.value(0.75)
).add_selection(
    highlight
).properties(
    width=750 , height=350
)
lines = base.mark_line().encode(
    size=alt.condition(~highlight, alt.value(0.5), alt.value(4))
).interactive()

lines + points

alt.LayerChart(...)

#### Mittlere Aufenthaltsdauer nach Stadtkreis

In [76]:
data2betested.columns

Index(['AlterV20ueber80Sort_noDM', 'AlterV20ueber80Cd_noDM', 'AlterV20ueber80Kurz_noDM', 'SexCd',
       'SexKurz', 'HerkunftCd', 'HerkunftLang', 'KreisCd', 'KreisLang', 'AufDauerP25',
       'AufDauerMedian', 'AufDauerP75', 'AufDauerMittel', 'Jahr', 'Jahr_str', 'Jahr_nbr'],
      dtype='object')

In [78]:
myAgg = data2betested.reset_index()\
    .groupby(['StichtagDatJahr','KreisCd', 'KreisLang']) \
    .agg(mean_WBev=('AufDauerMittel', 'mean')) \
    .sort_values('KreisLang', ascending=True) 

myAgg.reset_index().head(3)

,StichtagDatJahr,KreisCd,KreisLang,mean_WBev
0,1993-01-01,1,Kreis 1,9
1,2008-01-01,1,Kreis 1,14
2,1995-01-01,1,Kreis 1,10


In [79]:
myTitle="Mittlere Aufenthaltsdauer nach Stadtkreis, "+data_min_date + "- "+data_max_date

highlight = alt.selection(type='single', on='mouseover',
                          fields=['KreisLang'], nearest=True)
#x='date:StichtagDatJahr',
base = alt.Chart(myAgg.reset_index().query('mean_WBev>0').sort_values('KreisCd', ascending=True), title=myTitle).encode(
    x=alt.X('StichtagDatJahr', axis=alt.Axis(title='Stadtkreis'))# , axis=alt.Axis(format='%', title='percentage')
    , y=alt.X('mean_WBev', axis=alt.Axis(title='Mean Aufenthaltsdauer'))
    , color=alt.Color('KreisLang', legend=alt.Legend(title="Altersgruppen", orient="right"))  
    ,tooltip=['StichtagDatJahr', 'KreisLang','mean_WBev']    
)
points = base.mark_circle().encode(
    opacity=alt.value(0.75)
).add_selection(
    highlight
).properties(
    width=750 , height=350
)
lines = base.mark_line().encode(
    size=alt.condition(~highlight, alt.value(0.5), alt.value(4))
).interactive()

lines + points

alt.LayerChart(...)

**Sharepoint als gecheckt markieren!**

Record auf Sharepoint: **[Link](https://kollaboration.intranet.stzh.ch/orga/ssz-produkte/Lists/SASA_Outputs/DispForm.aspx?ID=497&ContentTypeId=0x0100988EAF029F1EFE4CA675F53C32A5D53D01006DBC563E6FBE9E4EB6FDC780799752E1)**